In [1]:
import sys
import os
sys.path.append(os.path.abspath('..'))

import pandas as pd
import numpy as np

from src.data.data_loader import (
    load_supervised_data,
    load_unsupervised_data,
    load_supervised_features,
    load_unsupervised_features
)

from src.preprocessing.data_preprocessing import PreProcessingEngine

from src.features.feature_engineering import FeatureEngineeringEngine
from src.features.build_features import FeatureBuilder

pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

The scope of this notebook is to convert raw supervised and unsupervised datasets into processed feature datasets

The workflow is split into 3 major layers:
* PreProcessingEngine to apply deterministic raw-data clearing
* FeatureEngineeringEngine to create financially meaningful credit-risk features
* FeatureBuilder to implement full feature engineering pipeline that includes loading raw data, applying preprocessing and feature engineering steps, and saving processed features locally

In [2]:
### loading raw data
# supervised dataset
supervised_raw = load_supervised_data()
# unsupervised dataset
unsupervised_raw = load_unsupervised_data()

print(f'Supervised dataset shape: {supervised_raw.shape}')
print(f'Unsupervised dataset shape: {unsupervised_raw.shape}')

Supervised dataset shape: (59477, 23)
Unsupervised dataset shape: (59477, 23)


In [3]:
### feature builder engine
# initialize preprocessor
preprocessor = PreProcessingEngine()
# initialize feature engineer
feature_engineer = FeatureEngineeringEngine()
# initialize feature builder
feature_builder = FeatureBuilder(
    preprocessor = preprocessor,
    feature_engineer = feature_engineer,
    drop_datetime_columns = True
)

# build supervised & unsupervised features, and store them locally
supervised_features, unsupervised_features = feature_builder.build_all_features(save = True)

print(f'Supervised features shape: {supervised_features.shape}')
print(f'Unsupervised features shape: {unsupervised_features.shape}')

Supervised features shape: (59472, 42)
Unsupervised features shape: (59472, 43)


In [4]:
### compare columns before & after feature engineering step
# supervised features
supervised_removed_cols = sorted(
    set(supervised_raw.columns) - set(supervised_features.columns)
)
supervised_added_cols = sorted(
    set(supervised_features.columns) - set(supervised_raw.columns)
)
# unsupervised features
unsupervised_removed_cols = sorted(
    set(unsupervised_raw.columns) - set(unsupervised_features.columns)
)
unsupervised_added_cols = sorted(
    set(unsupervised_features.columns) - set(unsupervised_raw.columns)
)

print('--- Supervised Features ---')
print('Removed columns:')
print(supervised_removed_cols)
print('\nAdded columns:')
print(supervised_added_cols)

print('\n--- Unsupervised Features ---')
print('Removed columns:')
print(unsupervised_removed_cols)
print('\nAdded columns:')
print(unsupervised_added_cols)

--- Supervised Features ---
Removed columns:
['DisbursalDate', 'Employee_code_ID', 'Mobile Avl Flag', 'Sanctioned Amount', 'State_ID', 'UniqueID', 'VoterID Flag']

Added columns:
['AccountsPerCreditHistoryMonth', 'ActiveAccountRatio', 'BalanceToDisbursedRatio', 'CreditHistoryAgeBand', 'CurrentBalancePositiveAmount', 'DaysSinceDisbursement', 'DisbursementMonth', 'DisbursementQuarter', 'DisbursementYear', 'FICO Rating', 'FICO Rating Ordinal', 'HasNegativeCurrentBalance', 'HasNoActiveAccounts', 'HasZeroCurrentBalance', 'InquiryIntensityBand', 'InquiryPerCreditHistoryMonth', 'InquiryToAccountRatio', 'InstalmentToDisbursedRatio', 'IsBureauExcluded', 'IsHighLTV', 'IsVeryHighLTV', 'Legit FICO Scores', 'Legit LTV', 'OverdueAccountRatio', 'RecentlyOpenedAccountRatio', 'RecentlyOpenedAccountsPerCreditHistoryMonth']

--- Unsupervised Features ---
Removed columns:
['DisbursalDate', 'Employee_code_ID', 'Mobile Avl Flag', 'Sanctioned Amount', 'State_ID', 'UniqueID', 'VoterID Flag']

Added columns:
[

In [5]:
### missing value summary after feature engineering -- supervised dataset
# supervised features
missing_summary = (
    supervised_features
    .isna()
    .mean()
    .sort_values(ascending = False)
    .to_frame(name = 'missing rate')
    .round(4)
)

missing_summary.head(10)

,missing rate
InstalmentToDisbursedRatio,0.1950
BalanceToDisbursedRatio,0.1870
Legit FICO Scores,0.1097
InquiryPerCreditHistoryMonth,0.0304
RecentlyOpenedAccountsPerCreditHistoryMonth,0.0304
AccountsPerCreditHistoryMonth,0.0304
ActiveAccountRatio,0.0037
Number of Active Accounts,0.0037
Legit LTV,0.0006
Current Balance Amount,0.0000


In [6]:
### load processed features for validation
# supervised features
df_sup_features = load_supervised_features()
# unsupervised features
df_unsup_features = load_unsupervised_features()

print(f'Supervised features shape: {df_sup_features.shape}')
print(f'Unsupervised features shape: {df_unsup_features.shape}')

print('--- Supervised Features ---')
display(df_sup_features.head())

print('--- Unsupervised Features ---')
display(df_unsup_features.head())

Supervised features shape: (59472, 42)
Unsupervised features shape: (59472, 43)
--- Supervised Features ---


,Loan To Value,Branch ID,Age,Employment Type,State,FICO Score,Number of Accounts,Number of Active Accounts,Number of Overdue Accounts,Current Balance Amount,Disbursed Amount,Instalment Amount,Number of Accounts Opened Last 6 Months,Average Account Age,Number of Inquiries,Target,DisbursementYear,DisbursementQuarter,DisbursementMonth,DaysSinceDisbursement,IsBureauExcluded,Legit FICO Scores,FICO Rating,FICO Rating Ordinal,HasNegativeCurrentBalance,HasZeroCurrentBalance,CurrentBalancePositiveAmount,InstalmentToDisbursedRatio,BalanceToDisbursedRatio,HasNoActiveAccounts,ActiveAccountRatio,OverdueAccountRatio,RecentlyOpenedAccountRatio,CreditHistoryAgeBand,AccountsPerCreditHistoryMonth,RecentlyOpenedAccountsPerCreditHistoryMonth,InquiryIntensityBand,InquiryPerCreditHistoryMonth,InquiryToAccountRatio,IsHighLTV,IsVeryHighLTV,Legit LTV
0,88.17,18,53,Self employed,Louisiana,681,9,3.0,0,63727,100984,5571,1,9,0,0,2018,3,8,0,0,681.0,Good,3,0,0,63727,0.055167,0.631060,0,0.333333,0.000000,0.111111,Young,1.000000,0.111111,No Inquiry,0.0,0.0,1,0,88.17
1,83.78,13,39,Salaried,Tennessee,384,3,1.0,1,103435,100000,160963,0,36,0,1,2018,3,8,0,0,384.0,Poor,1,0,0,103435,1.609630,1.034350,0,0.333333,0.333333,0.000000,Mature,0.083333,0.000000,No Inquiry,0.0,0.0,1,0,83.78
2,74.89,63,28,Salaried,New Jersey,673,8,2.0,0,120985,130600,15820,1,1,0,0,2018,3,8,0,0,673.0,Good,3,0,0,120985,0.121133,0.926378,0,0.250000,0.000000,0.125000,Very Young,8.000000,1.000000,No Inquiry,0.0,0.0,0,0,74.89
3,73.65,67,35,Self employed,Nevada,16,1,0.0,0,0,0,0,0,6,0,0,2018,3,8,0,1,NaN,Bureau Excluded,0,0,1,0,NaN,NaN,1,0.000000,0.000000,0.000000,Young,0.166667,0.000000,No Inquiry,0.0,0.0,0,0,73.65
4,83.29,16,32,Salaried,Illinois,626,6,2.0,0,42588,51320,7916,3,4,0,0,2018,3,8,0,0,626.0,Fair,2,0,0,42588,0.154248,0.829852,0,0.333333,0.000000,0.500000,Very Young,1.500000,0.750000,No Inquiry,0.0,0.0,1,0,83.29


--- Unsupervised Features ---


,Loan To Value,Branch ID,Age,Employment Type,State,FICO Score,Number of Accounts,Number of Active Accounts,Number of Overdue Accounts,Current Balance Amount,Disbursed Amount,Instalment Amount,Number of Accounts Opened Last 6 Months,Number of Delinquencies Last 6 Months,Average Account Age,Number of Inquiries,DisbursementYear,DisbursementQuarter,DisbursementMonth,DaysSinceDisbursement,IsBureauExcluded,Legit FICO Scores,FICO Rating,FICO Rating Ordinal,HasNegativeCurrentBalance,HasZeroCurrentBalance,CurrentBalancePositiveAmount,InstalmentToDisbursedRatio,BalanceToDisbursedRatio,HasNoActiveAccounts,ActiveAccountRatio,OverdueAccountRatio,RecentlyOpenedAccountRatio,RecentlyDelinquencyAccountRatio,CreditHistoryAgeBand,AccountsPerCreditHistoryMonth,RecentlyOpenedAccountsPerCreditHistoryMonth,InquiryIntensityBand,InquiryPerCreditHistoryMonth,InquiryToAccountRatio,IsHighLTV,IsVeryHighLTV,Legit LTV
0,73.23,67,33,Self employed,Nevada,598,1,1.0,1,27600,50200,1991,0,1,13,0,2018,3,9,56,0,598.0,Fair,2,0,0,27600,0.039661,0.549801,0,1.0,1.0,0.0,1.0,Established,0.076923,0.0,No Inquiry,0.000000,0.000000,0,0,73.23
1,88.48,67,25,Self employed,Nevada,305,3,0.0,0,0,0,31,0,0,8,1,2018,4,10,86,0,305.0,Poor,1,0,1,0,NaN,NaN,1,0.0,0.0,0.0,0.0,Young,0.375000,0.0,Single Inquiry,0.125000,0.333333,1,0,88.48
2,89.66,67,28,Self employed,Nevada,825,2,0.0,0,0,0,1347,0,0,21,0,2018,3,9,49,0,825.0,Exceptional,5,0,1,0,NaN,NaN,1,0.0,0.0,0.0,0.0,Established,0.095238,0.0,No Inquiry,0.000000,0.000000,1,0,89.66
3,71.89,67,29,Salaried,Nevada,17,1,1.0,0,72879,74500,0,0,0,2,0,2018,3,9,46,1,NaN,Bureau Excluded,0,0,0,72879,0.000000,0.978242,0,1.0,0.0,0.0,0.0,Very Young,0.500000,0.0,No Inquiry,0.000000,0.000000,0,0,71.89
4,89.56,67,27,Self employed,Nevada,718,1,1.0,0,-41,365384,0,0,0,56,1,2018,3,9,35,0,718.0,Good,3,1,0,0,0.000000,0.000000,0,1.0,0.0,0.0,0.0,Mature,0.017857,0.0,Single Inquiry,0.017857,1.000000,1,0,89.56
